<a href="https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb pandas pyarrow huggingface_hub

Load your Hugging Face token

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "HF_TOKEN not found.")

Token loaded successfully!


Connect DuckDB

In [4]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully!")

DuckDB connected successfully!


Load the HTTPFS extension

In [5]:
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

print("Extensions loaded!")

Extensions loaded!


Connect to the FlyRank warehouse

In [6]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [8]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [9]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_query_90d']}
LIMIT 5;
""").df()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [11]:
df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    ga4_engaged_sessions,

    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 100.0 / gsc_impressions
        ELSE 0
    END AS ctr

FROM {TABLES["fact_daily"]}

WHERE
    gsc_data_available IS TRUE
    AND month = '2026-03'
LIMIT 100000
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
df["reason_code"] = ""

df.loc[(df["gsc_impressions"] >= 1000) & (df["ctr"] < 1), "reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

df.loc[(df["gsc_avg_position"] > 10), "reason_code"] = "LOW_SEARCH_POSITION"

df.loc[(df["ga4_pageviews"] < 100), "reason_code"] = "LOW_PAGE_ENGAGEMENT"

In [13]:
action_map = {
    "HIGH_IMPRESSIONS_LOW_CTR": "Improve Title and Meta Description",
    "LOW_SEARCH_POSITION": "Improve SEO Content",
    "LOW_PAGE_ENGAGEMENT": "Enhance User Experience"
}

df["recommended_action"] = df["reason_code"].map(action_map)

In [14]:
queue = df.sort_values(
    by="gsc_impressions",
    ascending=False
)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended Use

This action playbook is designed to help SEO analysts prioritize content optimization opportunities identified by the machine learning model. The ranked recommendations are intended as decision-support rather than automatic decisions.

## Limits

- The recommendations are based on historical Google Search Console and GA4 data.
- The model was evaluated on the available dataset and may not generalize to all websites or future search trends.
- The action queue should be reviewed by a human before implementation.
- The recommendations are directional and should not be treated as guaranteed improvements.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human Review

Before implementing any recommendation, an SEO analyst should review:

- Search intent
- Content quality
- Business goals
- Brand guidelines
- Current events or seasonal trends
- Existing marketing strategy

## What Should NOT Be Automated

The following actions should always require human approval:

- Publishing new content
- Changing titles and meta descriptions automatically
- Deleting or redirecting pages
- Business-critical SEO decisions
- Changes that may affect legal, financial, or brand content

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring

The performance of the recommendation system should be monitored regularly using:

- Click-through Rate (CTR)
- Google Search impressions
- Average search position
- Organic sessions
- User engagement metrics

## Retrain Triggers

The model should be retrained when:

- CTR patterns change significantly.
- New clients or websites are added.
- Search engine algorithms change.
- Data schema changes.
- Model performance decreases over time.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/ranked_action_queue.csv",
    index=False
)

print("Ranked action queue exported successfully.")


Ranked action queue exported successfully.


## Self-check

Before you submit, confirm each line honestly:

- [x] Ranked actions created.
- [x] Reason codes assigned.
- [x] Intended use explained.
- [x] Human review process documented.
- [x] No-go cases identified.
- [x] Monitoring metrics defined.
- [x] Retrain triggers listed.
- [x] Ranked action queue exported.
- [x] Notebook runs successfully from top to bottom.